# n-Gram Language Models and Smoothing: From Laplace to Kneser–Ney

A language model assigns a probability to the next word given the words before it. Counting is the
obvious estimator and it fails on the first sentence it has not seen: a third of our held-out trigrams
never occur in training. Every smoother is a guess about that unseen mass. This notebook walks the
topic's argument with every number computed by `n_gram_models_smoothing.py`, which owns them; nothing
here redefines the math.

The arc reads smoothing three ways as empirical Bayes. A Dirichlet prior fit to the data by Type-II
maximum likelihood (§2); Good–Turing as Robbins' formula on word counts (§3); and, after backoff (§4),
absolute discounting (§5), and Kneser–Ney (§6), Kneser–Ney as an approximation to a hierarchical
Pitman–Yor posterior (§7). The finance case study (§8) asks what each estimate is an estimate *of*.

In [ ]:
import math
import sys
from pathlib import Path

here = Path.cwd()
sys.path.insert(0, str(here if (here / "n_gram_models_smoothing.py").exists()
                       else here / "notebooks" / "n-gram-models-smoothing"))
import n_gram_models_smoothing as ng

c = ng.train_counts()
print(f"training: {len(ng.corpus()['train'])} sentences, {sum(c[1].values())} predicted tokens "
      f"(words and </s>), |V| = {len(ng.vocabulary())} (training types, </s>, <unk>)")
print(f"held out: {len(ng.heldout_events())} events from generate_corpus(100, 1); "
      f"EDGAR: {len(ng.heldout_events('edgar'))} events")
for k in (1, 2, 3):
    print(f"distinct {k}-grams: {len(c[k]):6d}   N_1..N_5 = {[ng.count_of_counts(c[k]).get(r, 0) for r in range(1, 6)]}")

## 1. The zero-count problem

An **order-$n$ model** predicts token $w_i$ from the $n - 1$ tokens before it,
$P(w_i \mid w_{i-n+1}, \dots, w_{i-1})$; sentences are padded with $n - 1$ copies of `<s>` in front and one
`</s>` behind, which is predicted like a word. Write $u$ for a context (an $(n-1)$-gram), $c(u\,w)$ for
the count of the $n$-gram $u\,w$, and $c(u\,\cdot) = \sum_w c(u\,w)$.

**Proposition 1 (the MLE is the relative frequency).** Among all distributions $q$ on the vocabulary,
$\sum_w c(u\,w)\log q(w)$ is maximized by $\hat P(w \mid u) = c(u\,w)/c(u\,\cdot)$, and the gap to any $q$ is
exactly $c(u\,\cdot)\,\mathrm{KL}(\hat P \,\|\, q) \ge 0$ (Gibbs' inequality).

**Proposition 2 (the zero-frequency catastrophe).** The MLE's held-out log-loss
$-\frac1N \sum_i \log \hat P(w_i \mid u_i)$ is infinite iff some held-out $n$-gram never occurs in training.

In [ ]:
ng.test_counts_are_marginals()
ng.test_mle_maximizes_likelihood()
ng.test_zero_frequency_catastrophe()
for k, s in ng.unseen_shares().items():
    print(f"held-out events whose {k}-gram is unseen in training: {s:.2%}")
print("MLE held-out log-loss:", ng.log_loss(ng.mle_prob, ng.heldout_events()))

## 2. Smoothing as shrinkage

Picture the MLE as a point in the probability simplex $\Delta^{|V|-1}$. For a context seen a handful of
times it sits on a low-dimensional face: every unseen word has coordinate 0. Smoothing moves it into the
interior. The **hierarchical Dirichlet** estimate (MacKay and Peto 1995) moves it along a straight line
toward the next-lower-order model $P(\cdot \mid u^-)$, where $u^-$ drops the oldest word of $u$:

$$P(w \mid u) = \frac{c(u\,w) + \alpha\,P(w \mid u^-)}{c(u\,\cdot) + \alpha}, \qquad P(w) = \frac{c(w) + \alpha_1/|V|}{N + \alpha_1}.$$

It is the posterior mean of the next-word distribution under a $\mathrm{Dirichlet}(\alpha\,P(\cdot\mid u^-))$
prior (formalRAG `query-likelihood-language-models`, Theorem 1); with a uniform base it is add-$\delta$
(Lidstone), $\delta = \alpha/|V|$, and Laplace at $\delta = 1$.

**Proposition 3 (shrinkage is interpolation).** $P(w \mid u) = (1 - \lambda_u)\,\hat P(w \mid u) + \lambda_u\,P(w \mid u^-)$
with $\lambda_u = \alpha/(c(u\,\cdot) + \alpha)$: rarely seen contexts are pulled harder, unseen ones all the way.

**Theorem 1 (the evidence).** Under independent $\mathrm{Dirichlet}(\alpha m_u)$ priors per context, each centered on
its own $m_u = P(\cdot \mid u^-)$, the probability of the training word sequences is
$$\log p(\text{data} \mid \alpha) = \sum_u \Big[\log\Gamma(\alpha) - \log\Gamma(\alpha + c(u\,\cdot))\Big] + \sum_{u,w} \Big[\log\Gamma(\alpha m_{uw} + c(u\,w)) - \log\Gamma(\alpha m_{uw})\Big].$$
Maximizing it over $\alpha$ is **Type-II maximum likelihood** — empirical Bayes with a parametric prior
(formalStatistics `hierarchical-bayes-and-partial-pooling` §28.7).

In [ ]:
ng.test_dirichlet_is_interpolation()
ng.test_evidence_matches_scipy()
ng.test_type2_alpha_is_a_maximum()
print("Type-II alphas (unigram, bigram, trigram):", [round(a, 2) for a in ng.type2_alphas()])
print("held-out-optimal alphas (oracle):        ", [round(a, 2) for a in ng.heldout_optimal_alphas()])
print(f"held-out log-loss: Laplace {ng.log_loss(ng.additive_prob, ng.heldout_events()):.4f}, "
      f"Dirichlet Type-II {ng.dirichlet_log_loss(ng.type2_alphas()):.4f}, "
      f"Dirichlet oracle {ng.dirichlet_log_loss(ng.heldout_optimal_alphas()):.4f} nats/token")

The evidence never sees a word it cannot predict: training text has no `<unk>` events. So at the unigram and
bigram orders the Type-II $\hat\alpha$ under-smooths for a held-out set in which 0.67% of tokens are new
words, and the oracle — which peeks at the held-out data — smooths about sixty-two times harder there. At the
trigram order the comparison reverses: the oracle's $\alpha$ is *smaller* than Type-II's.

## 3. Good–Turing: Robbins' formula on word counts

Let $N_r$ be the number of distinct types seen exactly $r$ times in a sample of $n$ tokens, and $M_r$ the
total true probability of those types. Good's (1953) estimate, credited to Turing, replaces a count $r$ by
$$r^* = (r + 1)\,\frac{N_{r+1}}{N_r}, \qquad \text{and the unseen types share } \frac{N_1}{n}.$$

**Theorem 2.** For $n$ i.i.d. draws from any $p$, $\;\mathbb E[M_r(n)] = \frac{r+1}{n+1}\,\mathbb E[N_{r+1}(n+1)]$ exactly.
*(Expand both sides over types; $\binom{n+1}{r+1} = \frac{n+1}{r+1}\binom{n}{r}$ matches them term by term.)*

**Corollary.** $0 \le \mathbb E[N_1(n)]/n - \mathbb E[M_0(n)] = \sum_w p_w^2(1 - p_w)^{n-1} \le 1/n$.

Poissonize the counts, $c_w \sim \mathrm{Poisson}(\lambda_w)$, and treat the spread of $\lambda_w$ over the
vocabulary as a prior $g$: then $\mathbb E[\lambda \mid c = r] = (r+1)f(r+1)/f(r)$ — **Robbins' formula**
(formalStatistics `empirical-bayes` §33.3, Theorem 2), with types in place of policy holders and $N_r$ in
place of the claim counts. Turing's $r^*$ is Robbins' estimator.

In [ ]:
ng.test_good_turing_identity_exact()
ng.test_missing_mass_bias_bound()
ng.test_good_turing_is_robbins()
casi = ng.turing_rstar(dict(enumerate(ng.CASI_CLAIMS)))
print("Turing's r* on Topic 33's insurance claims:", [round(casi[x], 3) for x in range(7)])
print("Topic 33's printed Robbins row:            ", list(ng.CASI_ROBBINS))

Robbins' estimate is erratic where $N_{r+1}$ is small (Topic 33, Remark 5), and so is Turing's: on our
trigrams $r - r^*$ even turns negative at $r = 6$. **Simple Good–Turing** (Gale and Sampson 1995) fits
$\log Z_r = a + b\log r$ to the averaged spectrum and switches from Turing's $r^*$ to the fitted one once the
two agree within 1.96 standard errors. The log-linear fit is a heuristic, made plausible by Topic 1's
power-law spectrum. Its discounts $r - r^*$ are nearly constant — the observation behind absolute discounting.

In [ ]:
ng.test_sgt_matches_nltk()
N3 = ng.count_of_counts(c[3])
raw, sgt = ng.turing_rstar(N3), ng.simple_good_turing(N3)
print(f"SGT on trigrams: slope {sgt.slope:.3f}, switch at r = {sgt.switch_at}")
for r in range(1, 9):
    print(f"r = {r}:  Turing r - r* = {r - raw[r]:7.3f}   SGT r - r* = {r - sgt.rstar(r):.3f}")

## 4. Backoff

**Katz backoff** (1987) keeps a seen $n$-gram's Good–Turing-discounted relative frequency and sends an
unseen one to the lower order, scaled by a backoff weight:
$$P_{\text{katz}}(w \mid u) = \begin{cases} d_{c(u w)}\,c(u\,w)/c(u\,\cdot) & c(u\,w) > 0 \\ \alpha(u)\,P_{\text{katz}}(w \mid u^-) & \text{otherwise,} \end{cases}$$
with $d_r$ from Turing's $r^*$ for $r \le k = 5$ and $d_r = 1$ above.

**Proposition 4.** $\alpha(u) = \dfrac{1 - \sum_{w \text{ seen}} d\,c(u\,w)/c(u\,\cdot)}{1 - \sum_{w \text{ seen}} P_{\text{katz}}(w \mid u^-)}$
makes $P_{\text{katz}}(\cdot \mid u)$ sum to one whenever the denominator is positive.

The cutoff has a cost on templated text: a context whose followers were all seen more than $k$ times
reserves no mass at all, and the zero-count problem comes back.

In [ ]:
ng.test_katz_normalizes()
ng.test_katz_cutoff_reintroduces_zeros()
km = ng.katz_model()
print("Katz discounts d_r (bigram, trigram):", {k: {r: round(v, 3) for r, v in d.items()} for k, d in km.d.items()})
print("contexts reserving no mass:", sum(a == 0 for a in km.alpha.values()), "of", len(km.alpha))
print("held-out events Katz gives probability 0:", ng.katz_zero_events())

## 5. Absolute discounting

Subtract one constant $D \in (0, 1]$ from every seen count and interpolate with the lower order:
$$P(w \mid u) = \frac{\max(c(u\,w) - D, 0) + D\,N_{1+}(u\,\bullet)\,P(w \mid u^-)}{c(u\,\cdot)},$$
where $N_{1+}(u\,\bullet)$ is the number of distinct words seen after $u$. It normalizes (Proposition 5):
the subtracted mass $D\,N_{1+}(u\,\bullet)$ is exactly what the lower order receives.

**Theorem 3 (the leave-one-out discount).** For the backing-off form with a fixed lower order, the
leave-one-out log-likelihood is $F(D) = n_1\log D + \sum_{r \ge 2} r\,n_r\log(r - 1 - D) + C$ (with $n_1$ the
singletons whose context survives their removal). $F$ is strictly concave on $(0, 1)$, has a unique
maximizer $D^*$, and $D^* \le n_1/(n_1 + 2n_2)$. *(At $D^*$, $n_1/D^* = \sum_{r\ge2} r n_r/(r-1-D^*) \ge 2n_2/(1-D^*)$.)*
Ney, Essen, and Kneser's familiar estimate is this upper bound.

In [ ]:
ng.test_loo_concave_and_bound()  # Prop 5 (normalization) is in test_ikn_normalizes, §6
ng.test_loo_objective_is_exact()
for k in (2, 3):
    N = ng.loo_spectrum(c[k])
    print(f"order {k}: leave-one-out D* = {ng.loo_discount(N):.4f}   bound n1/(n1 + 2 n2) = "
          f"{N[1] / (N[1] + 2 * N[2]):.4f}   Ney's (all singletons) = {ng.ney_discount(ng.count_of_counts(c[k])):.4f}")

## 6. Kneser–Ney

`industrial` occurs 284 times — always right after `halvorsen`. Absolute discounting's lower order
believes it is a common word; as the continuation of an unseen context it is almost never right. Kneser
and Ney (1995) replace the lower-order counts with **continuation counts**
$N_{1+}(\bullet\,v) = |\{x : c(x\,v) > 0\}|$, the number of distinct contexts $v$ completes.

**Theorem 4 (the marginal constraint).** For the interpolated model with discounts $D(c)$ at the top
order, the lower-order distribution $\beta$ that preserves the empirical marginals,
$\sum_x c(x\,v\,\cdot)\,P(w \mid x\,v) = c(v\,w)$ for all $v, w$, is unique:
$\beta(w \mid v) \propto \sum_x D(c(x\,v\,w))$. With one discount per order this is
$N_{1+}(\bullet\,v\,w)/N_{1+}(\bullet\,v\,\bullet)$ — the continuation distribution.

**Interpolated Kneser–Ney** applies this at every order: raw counts at the top, continuation counts
below, one discount per order, and a uniform distribution under the unigram level. **Modified KN**
(Chen and Goodman 1998) uses three discounts $D_1, D_2, D_{3+}$ per order — a heuristic, and by Theorem 4
its continuation-count lower orders no longer preserve the marginals exactly.

In [ ]:
ng.test_continuation_counts_bruteforce()
ng.test_ikn_normalizes()
ng.test_kn_marginal_constraint()
ng.test_marginals_need_discount_weighted_lower_order()
ng.test_ikn_matches_nltk()
print("frequent words that continue few contexts (word, count, N1+(. w)):", ng.raw_vs_continuation())
print("Ney discounts per order:", {k: round(v[0], 4) for k, v in ng.kn_models()["ikn"].discounts.items()})
print("modified-KN discounts per order:", {k: tuple(round(x, 3) for x in v) for k, v in ng.kn_models()["mkn"].discounts.items()})

## 7. Kneser–Ney as a hierarchical Pitman–Yor posterior

Topic 1 drew word types from a Pitman–Yor Chinese restaurant with discount $d$ and strength $\theta$. The
**hierarchical Pitman–Yor language model** (Teh 2006) puts one restaurant in every context $u$, whose base
distribution is the restaurant of $u^-$: each table in $u$ sends one customer to $u^-$. Given a seating
with $c_{uw}$ customers and $t_{uw}$ tables for word $w$,
$$P(w \mid u) = \frac{c_{uw} - d_k t_{uw}}{\theta_k + c_{u\cdot}} + \frac{\theta_k + d_k t_{u\cdot}}{\theta_k + c_{u\cdot}}\,P(w \mid u^-).$$

**Proposition 6.** Seat every word at exactly one table, $t_{uw} = \min(1, c_{uw})$, and set $\theta_k = 0$. Then
the parent restaurants' customer counts are the continuation counts, and the predictive is interpolated
Kneser–Ney with $D_k = d_k$, at every order.

In [ ]:
ng.test_hpylm_one_table_is_ikn()
seat = ng.one_table_seating(ng.corpus()["train"])
print("unigram restaurant customers for 'industrial':", seat.customers[()]["industrial"],
      "(its raw count is", c[1][("industrial",)], ")")

One table per type is one seating among many. Teh's Gibbs sampler re-seats every training token in turn
and resamples $(d_k, \theta_k)$ from their posterior. Starting from the Kneser–Ney seating, the sampler
finds frequent $n$-grams spread over several tables, so their effective discount $d\,\mathbb E[t \mid c = r]$
grows with $r$ — far beyond modified KN's three steps — while the higher-order $\theta_k$ settle near 0,
Kneser–Ney's pure-discounting regime. It is MCMC: we report averages over sweeps 21–40 and claim no
convergence guarantee.

In [ ]:
h = ng.hpylm_run()
ng.test_hpylm_sampler_invariants()
ng.test_hpylm_discount_grows_with_count()
print("posterior means d:", {k: round(v, 3) for k, v in h["d"].items()}, " theta:", {k: round(v, 4) for k, v in h["theta"].items()})
print("effective trigram discount d·E[t | r], r = 1..8:", [round(x, 3) for x in h["effective_discount"]])
print("modified KN's trigram discounts:", [round(x, 3) for x in ng.kn_models()["mkn"].discounts[3]])

## The smoothers side by side

Held-out log-loss in nats per token (the next topic, `perplexity-cross-entropy`, turns it into
perplexity). In domain the ordering is the textbook one, with the sampled HPYLM on top. Out of domain —
three EDGAR MD&A excerpts, where 58% of tokens are words the model has never seen — the heavy uniform
smoothing of Laplace wins.

In [ ]:
ng.test_heldout_ordering()
t = ng.smoother_table()
for name, row in t.items():
    print(f"{name:18s}", "  ".join(f"{test} {v:7.4f}" for test, v in row.items()))
zeros = set(ng.katz_zero_events())
print(f"Katz on the {len(ng.heldout_events()) - len(zeros)} held-out events it does not zero out:",
      round(ng.log_loss(ng.prob_fn("katz"), [g for g in ng.heldout_events() if g not in zeros]), 4))

## 8. Finance case study: what is each estimate an estimate of?

Two lessons from the shared synthetic earnings-call corpus.

**The Halvorsen effect.** Company names are fixed phrases. `industrial` is a top-30 word to absolute
discounting's unigram level and ranks below 1,000 to Kneser–Ney's, because it continues one context.

**Good–Turing forecasts the continuation.** $N_1/N$ estimates the probability that the *next* token from
the same process is a new word. Run the training corpus's own generator on for 100 more documents and
the forecast holds; a fresh run of the generator restarts its Pitman–Yor restaurants from the same seed
nouns and sees far fewer new words; real MD&A text is another source and sees twenty times more.

In [ ]:
ng.test_continuation_rebuild_matches_corpus()
ng.test_good_turing_forecasts_the_continuation()
ng.test_halvorsen_effect()
f = ng.finance_summary()
print(f"'industrial': {f['halvorsen_raw']} tokens, {f['halvorsen_continuation']} context; unigram rank "
      f"{f['rank_absdisc']} (absolute discounting) vs {f['rank_ikn']} (Kneser-Ney)")
print(f"Good-Turing unseen mass N1/N = {f['gt_unseen_mass']:.4f}")
print(f"OOV share: continuation {f['oov_continuation']:.4f}, fresh run {f['oov_fresh_run']:.4f}, EDGAR {f['oov_edgar']:.4f}")

## 9. The lab's numbers, and the full harness

`viz_payload()` holds every number the interactive lab displays: Panel A's next-word distributions for
five contexts under every smoother (and every slider value of $D$), with held-out log-loss; Panel B's
discount-versus-count curves and the trigram spectrum.

In [ ]:
ng.test_viz_payload()
P = ng.viz_payload()
for ctx in P["contexts"]:
    d = ctx["by_d"]["ikn"][6]
    print(" ".join(ctx["context"]), "___", f"(seen {ctx['count']}x):", [w for w, _, _ in d["top"][:4]],
          f"unseen mass {d['unseen_mass']:.3f}")
print("IKN held-out log-loss by D:", dict(zip(P["D"], P["logloss"]["by_d"]["ikn"])))
print("Panel B:", P["discount"])

In [ ]:
assert ng._run_harness() == 0